# Dataset Creation and Visualization
Visualize a dataset containing two modalities: RGB and Lidar of spheres and cubes.
This is adjusted from the NVIDIA Multi Modal course.

In [1]:
# Install uv for fast package management
!pip install uv

# Install required libraries
!uv pip install fiftyone huggingface_hub gdown ipywidgets numpy matplotlib tqdm

Using Python 3.11.11 environment at: C:\Users\Philipp\2_uni\wise2526\AHOCV\Applied-Hands-On-Computer-Vision\.venv
Audited 7 packages in 121ms


Specify dataset names, google drive directory (data source) and huggingface data (data repository). 

In [2]:
import fiftyone as fo
import os
import glob
import gdown
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Configuration
DATA_DIR = "data/assessment"
DRIVE_FOLDER_ID = "1ZWWafNF6ljLJN3R1_ABzlfaG8i0jc_O0"
SUBSET_SIZE = 750 # Increased to 1000 to have enough for train/val/test
DATASET_NAME = "cilp_assessment"
SUBSET_NAME = "cilp_assessment_subset"
# Replace with your Hugging Face username or set via environment variable
HF_USERNAME = "philippkolbe" 
HF_DATASET_REPO = f"{HF_USERNAME}/{SUBSET_NAME}"

print(f"Using data directory: {os.path.abspath(DATA_DIR)}")

Using data directory: c:\Users\Philipp\2_uni\wise2526\AHOCV\Applied-Hands-On-Computer-Vision\assignment2\data\assessment


## Load Dataset

Load data from provided google drive. If it does not work we need to load it manually.

In [3]:
# Download dataset from Google Drive
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"Downloading data from Drive ID: {DRIVE_FOLDER_ID}...")
    try:
        # Try gdown first
        gdown.download_folder(id=DRIVE_FOLDER_ID, output=DATA_DIR, quiet=False, use_cookies=False)
        print("Download complete.")
    except Exception as e:
        print(f"Gdown download failed: {e}")
        print("\n--- TROUBLESHOOTING ---")
        print("The folder has too many files for direct download via gdown.")
        print("1. If you are in Google Colab:")
        print(f"   a. Open the Drive link: https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
        print("   b. Click 'Organize' -> 'Add shortcut' -> 'My Drive'")
        print("   c. Mount Drive and copy data:")
        print("      from google.colab import drive")
        print("      drive.mount('/content/drive')")
        print(f"      !cp -r /content/drive/MyDrive/ShortcutName/* {DATA_DIR}")
        print("\n2. If you are running locally:")
        print(f"   a. Download the folder manually from: https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
        print(f"   b. Extract/Place the 'cubes' and 'spheres' folders into: {os.path.abspath(DATA_DIR)}")
else:
    print(f"Data directory {DATA_DIR} already exists. Skipping download.")

Data directory data/assessment already exists. Skipping download.


## Create FiftyOne Dataset

Create FiftyOne dataset with all available data. Create groups (modality rgb or lidar) and classifications (sphere or square).

In [4]:
# Create FiftyOne Dataset
if fo.dataset_exists(DATASET_NAME):
    print(f"Dataset {DATASET_NAME} already exists. Deleting...")
    fo.delete_dataset(DATASET_NAME)

dataset = fo.Dataset(DATASET_NAME)
dataset.add_group_field("group", default="rgb")

samples = []
classes = ["cubes", "spheres"]

print("Creating dataset samples...")
for label in classes:
    rgb_dir = os.path.join(DATA_DIR, label, "rgb")
    lidar_dir = os.path.join(DATA_DIR, label, "lidar")
    
    if not os.path.exists(rgb_dir):
        print(f"Warning: {rgb_dir} not found. Skipping {label}.")
        continue

    rgb_images = glob.glob(os.path.join(rgb_dir, "*.png"))
    skipped_count = 0
    
    # Use tqdm for progress bar
    for rgb_path in tqdm(rgb_images, desc=f"Processing {label}"):
        filename = os.path.basename(rgb_path)
        file_id = os.path.splitext(filename)[0]
        
        # Construct LiDAR path
        lidar_npy_path = os.path.join(lidar_dir, f"{file_id}.npy")
        
        if os.path.exists(lidar_npy_path):
            # Create Group
            group = fo.Group()
            
            # RGB Sample
            rgb_sample = fo.Sample(filepath=rgb_path, group=group.element("rgb"))
            rgb_sample["ground_truth"] = fo.Classification(label=label)
            
            # LiDAR Sample - Point to NPY initially (visualization generated later for subset)
            lidar_sample = fo.Sample(filepath=lidar_npy_path, group=group.element("lidar"))
            lidar_sample["ground_truth"] = fo.Classification(label=label)
            lidar_sample["raw_filepath"] = lidar_npy_path 
            
            samples.append(rgb_sample)
            samples.append(lidar_sample)
        else:
            skipped_count += 1

    print(f"Processed {label}: Found {len(rgb_images)} RGB images. Paired {len(rgb_images) - skipped_count}. Skipped {skipped_count} missing LiDAR.")

if samples:
    dataset.add_samples(samples)
    print(f"Created dataset '{DATASET_NAME}' with {len(dataset)} samples (including both modalities).")
else:
    print("No samples found! Check data directory structure.")

Dataset cilp_assessment already exists. Deleting...
Creating dataset samples...


Processing cubes: 100%|██████████| 9999/9999 [00:11<00:00, 880.86it/s] 


Processed cubes: Found 9999 RGB images. Paired 9999. Skipped 0 missing LiDAR.


Processing spheres: 100%|██████████| 752/752 [00:01<00:00, 592.10it/s]

Processed spheres: Found 752 RGB images. Paired 752. Skipped 0 missing LiDAR.


 100% |█████████████| 21502/21502 [11.6s elapsed, 0s remaining, 1.9K samples/s]      
Created dataset 'cilp_assessment' with 10751 samples (including both modalities).


Launch app to take a look at the raw data.

In [6]:
session = fo.launch_app(dataset)

## Subset Creation, Visualization and Exploration

Visualize lidar data as a png image.

In [7]:
def save_lidar_vis(npy_path, save_dir):
    """Converts .npy LiDAR data to a .png heatmap for visualization."""
    try:
        data = np.load(npy_path)
        
        # Normalize to 0-1 for visualization
        if data.max() - data.min() != 0:
            data_norm = (data - data.min()) / (data.max() - data.min())
        else:
            data_norm = data
        
        filename = os.path.basename(npy_path).replace(".npy", ".png")
        save_path = os.path.join(save_dir, filename)
        
        # Save as heatmap
        plt.imsave(save_path, data_norm, cmap='viridis')
        return save_path
    except Exception as e:
        print(f"Error converting {npy_path}: {e}")
        return None

**Subset Creation**: Created a subset of 500 training samples, 100 validation and 100 test samples. The subset is balanced between classes and modalities.

**LiDAR Visualization:** We create another group element for the LiDAR data visualization as png images to make it easier to explore the data in FiftyOne.


In [8]:
import fiftyone.utils.random as four

# Create a balanced subset and upload
print("Creating balanced subset...")
# Calculate samples per class
n_per_class = SUBSET_SIZE // 2

# Select random samples from each class
cubes_view = dataset.match(fo.ViewField("ground_truth.label") == "cubes").shuffle(seed=12).limit(n_per_class)
spheres_view = dataset.match(fo.ViewField("ground_truth.label") == "spheres").shuffle(seed=12).limit(n_per_class)

print(f"Selected {len(cubes_view)} cubes and {len(spheres_view)} spheres.")

# Combine into one view
subset_ids = cubes_view.values("id") + spheres_view.values("id")
subset_view = dataset.select(subset_ids).shuffle(seed=12)

print(f"Created balanced subset view with {len(subset_view)} samples.")

# Use a temporary name for the local clone so we don't conflict with the main dataset
if fo.dataset_exists(SUBSET_NAME):
    fo.delete_dataset(SUBSET_NAME)

print(f"Cloning view to temporary dataset '{SUBSET_NAME}'...")
subset_dataset = subset_view.clone(SUBSET_NAME)

print("Generating LiDAR visualizations for subset...")

# Iterate over the subset dataset to generate visualizations
# We use a separate directory for subset visualizations to keep things clean

# FIX: Explicitly select the 'lidar' slice so we iterate over LiDAR samples
lidar_view = subset_dataset.select_group_slices("lidar")
lidar_vis_samples = []

for sample in tqdm(lidar_view, desc="Visualizing LiDAR"):
    npy_path = sample.filepath
    # Only convert if it's an npy file (sanity check)
    if npy_path.endswith('.npy'):
        label = sample.ground_truth.label
        save_dir = os.path.join(DATA_DIR, label, "lidar_vis_subset")
        os.makedirs(save_dir, exist_ok=True)
        
        png_path = save_lidar_vis(npy_path, save_dir)

        if png_path:
            # Create a NEW sample for the visualization slice
            # This avoids MediaTypeError by keeping the original 'lidar' sample as-is (unknown/npy)
            # and creating a new 'lidar_vis' sample which is an image.
            vis_sample = fo.Sample(filepath=png_path, group=sample.group.element("lidar_vis"))
            vis_sample["ground_truth"] = sample.ground_truth            
            lidar_vis_samples.append(vis_sample)

if lidar_vis_samples:
    print(f"Adding {len(lidar_vis_samples)} visualization samples...")
    subset_dataset.add_samples(lidar_vis_samples)

print("Computing metadata...")
subset_dataset.compute_metadata()

# Create Splits (Train/Val/Test)
print("Applying random splits (66/16/16)...")
four.random_split(subset_dataset, {"train": 0.66, "val": 0.16, "test": 0.16}, seed=12)
print("Completed processing")

Creating balanced subset...
Selected 375 cubes and 375 spheres.
Created balanced subset view with 750 samples.
Cloning view to temporary dataset 'cilp_assessment_subset'...
Generating LiDAR visualizations for subset...


Visualizing LiDAR: 100%|██████████| 750/750 [00:12<00:00, 59.10it/s]

Adding 750 visualization samples...
   0% ||----------------|   1/750 [5.5ms elapsed, 4.2s remaining, 180.3 samples/s] 

 100% |█████████████████| 750/750 [479.2ms elapsed, 0s remaining, 1.6K samples/s]      
Computing metadata...
Computing metadata...
 100% |███████████████| 2250/2250 [3.7s elapsed, 0s remaining, 321.7 samples/s]        
Applying random splits (66/16/16)...
Completed processing


Check for duplicates

In [17]:
import hashlib
import fiftyone as fo

print("Checking for content duplicates...")

def get_file_hash(filepath):
    """Computes MD5 hash of a file."""
    hasher = hashlib.md5()
    try:
        with open(filepath, 'rb') as f:
            # Read in chunks to handle large files
            for chunk in iter(lambda: f.read(4096), b""):
                hasher.update(chunk)
        return hasher.hexdigest()
    except Exception:
        return None

def check_content_duplicates(slice_name):
    print(f"Checking '{slice_name}' slice for content duplicates...")
    # Select the specific group slice
    view = subset_dataset.select_group_slices(slice_name)
    
    seen_hashes = {}
    duplicates = []
    
    # Iterate and hash
    for sample in tqdm(view, desc=f"Hashing {slice_name}"):
        filepath = sample.filepath
        if not os.path.exists(filepath):
            continue
            
        file_hash = get_file_hash(filepath)
        if file_hash is None:
            continue
            
        if file_hash in seen_hashes:
            duplicates.append(sample)
            print(f"Found duplicate: {os.path.basename(filepath)} matches {os.path.basename(seen_hashes[file_hash])}")
        else:
            seen_hashes[file_hash] = sample.filepath
            
    print(f"  Found {len(duplicates)} duplicate content samples in '{slice_name}'.")

    return duplicates

# Check RGB
check_content_duplicates("rgb")

# Check LiDAR
check_content_duplicates("lidar")

print("Duplicate content check complete.")

Checking for content duplicates...
Checking 'rgb' slice for content duplicates...


Hashing rgb:  62%|██████▏   | 462/750 [00:00<00:00, 1125.85it/s]

Found duplicate: 2002.png matches 5910.png


Hashing rgb: 100%|██████████| 750/750 [00:00<00:00, 919.24it/s] 


  Found 1 duplicate content samples in 'rgb'.
Checking 'lidar' slice for content duplicates...


Hashing lidar: 100%|██████████| 750/750 [00:00<00:00, 875.59it/s] 

  Found 0 duplicate content samples in 'lidar'.
Duplicate content check complete.


The duplicate above are two fully black images. This is expected because of very unlucky lighting conditions during data capture. But LiDAR data has no duplicates, so we can keep both rgb images.

Compute some observations about the dataset:

In [10]:
# Calculate Statistics
print(f"\n--- Dataset Statistics {subset_dataset.name} ---")

# Total samples per class
print("\nClass Counts:")
print(subset_dataset.count_values("ground_truth.label"))

# Split sizes
print("\nSplit Sizes:")
print(subset_dataset.count_values("tags"))

# Class distribution per split
print("\nClass Distribution per Split:")
for tag in ["train", "val", "test"]:
    view = subset_dataset.match_tags(tag)
    print(f"  {tag}: {view.count_values('ground_truth.label')}")


--- Dataset Statistics cilp_assessment_subset ---

Class Counts:
{'cubes': 375, 'spheres': 375}

Split Sizes:
{'train': 505, 'test': 122, 'val': 123}

Class Distribution per Split:
  train: {'cubes': 260, 'spheres': 245}
  val: {'cubes': 58, 'spheres': 65}
  test: {'cubes': 57, 'spheres': 65}


Do some more manual exploration of the dataset in FiftyOne.

In [11]:
session.refresh()
print(session.url)

http://localhost:5151/


## Upload dataset to huggingface

Log in to huggingface to upload dataset there.

In [12]:
from huggingface_hub import login

# Login to Hugging Face
print("Logging in to Hugging Face...")
login()

Logging in to Hugging Face...


In [18]:
import fiftyone.utils.huggingface as fouh

# We upload to the repo name defined in configuration (SUBSET_NAME)
print(f"\nUploading dataset to {HF_DATASET_REPO}...")
try:
    fouh.push_to_hub(
        subset_dataset, 
        repo_name=SUBSET_NAME, 
        repo_type="dataset",
        private=False,
        exist_ok=True,
        chunk_size=100,
    )
    print("Upload successful!")
except Exception as e:
    print(f"Upload failed: {e}")
    print("Ensure you have write access to the repository and are logged in.")
    raise e


Uploading dataset to philippkolbe/cilp_assessment_subset...
Directory 'C:\Users\Philipp\AppData\Local\Temp\tmpku79qxfm' already exists; export will be merged with existing files
Exporting samples...
 100% |██████████████████| 2250/2250 [6.9s elapsed, 0s remaining, 448.8 docs/s]       


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
Uploading media files in 23 batches of size 100:   0%|          | 0/23 [00:00<?, ?it/s]No files have been modified since last commit. Skipping to prevent empty commit.
Uploading media files in 23 batches of size 100:   4%|▍         | 1/23 [00:03<01:15,  3.42s/it]No files have been modified since last commit. Skipping to prevent empty commit.
Uploading media files in 23 batches of size 100:   9%|▊         | 2/23 [00:07<01:15,  3.62s/it]No files have been modified since last commit. Skipping to prevent empty commit.
Uploading media files in 23 batches of size 100: 100%|██████████| 23/23 [01:57<00:00,  5.09s/it]
No files have been modified since last commit. Skipping to prevent empty commit.


Upload successful!


## Deliverables

Uploaded subset to [huggingface](https://huggingface.co/datasets/philippkolbe/cilp_assessment_subset).

### 1. Statistics

#### Total number of samples per class
- Originally, we had 9999 cubes but only 752 valid spheres (lidar missing)
- in our subset we choose 375 cubes, 375 spheres

#### Train/Validation/Test split sizes
- 750 subset samples
- ca. 500 train (66%), 125 val (16%), 125 test (16%)
- specifically: 505 train, 122 val, 123 test

#### Image dimensions and data types
- `rgb`: 64 x 64 x 4 channels (`image/png`)
- `lidar`: 16.512 bytes (`application/optec-stream`, `.npy` file)
- `lidar_vis`: 64 x 64 x 4 channels (`image/png`) -> just for visualization

#### Class Distrbution Histogram
![media/fiftyone_class_distribution.png](media/fiftyone_class_distribution.png)

### 2. FiftyOne Interface Screenshots
![media/comparison.png](media/comparison.png)
![media/fiftyone_overview_png.png](media/fiftyone_overview_png.png)
![media/fiftyone_overview_lidar_vis.png](media/fiftyone_overview_lidar_vis.png)

### 3. Dataset Observations
1. **Observation 1**: Class Imbalance: only 752 valid spheres available because many lidar data points are missing. In subset this is not a problem anymore.
2. **Observation 2**: Some cubes do not look like cubes at all when only looking at RGB data. Often they are very skewed due to lighting. But the LiDAR representations are a lot better to detect.
3. **Observation 3**: Some RGB images are completely black. Not sure why this is the case, likely due to very unlucky lighting conditions during data capture.
4. **Observation 4**: Alignment between LiDAR visualization and RGB seems to be off. This was already discussed in the NVIDIA notebook. The model should still be able to handle it, since it is trained on that data.

### 4. Data Quality Issues
- **Missing rgb sphere data:** There are as expected 10000 different samples for lidar spheres but way less rgb data. Since we are using a subset anyways it is not a problem.
- **Duplicates in lidar sphere data:** Lidar sphere data seems to have some copies called `(1).npy`. Not sure why this is in the original data. The only duplicates found in the subset were due to completely black rgb images. Their LiDAR data had no duplicates, so we kept both rgb images in the set because they are valid samples.
- **Black rgb images:** Some rgb images are completely black. Not sure if this is intended or a data collection issue. Likely due to very unlucky lighting conditions during data capture.